# GROUP BY, CTEs, and Subqueries

This notebook teaches three powerful SQL concepts through hands-on practice:

1. **GROUP BY** - Aggregate data into summary statistics
2. **CTEs (Common Table Expressions)** - Write readable, modular queries with `WITH` clauses
3. **Subqueries** - Nest queries inside queries for complex logic

We will build from simple aggregations up to a realistic multi-CTE business report.

---

## Setup: Helper Function

We use a `run_query` helper that connects to our PostgreSQL database, executes a query, and prints the results as a formatted table.

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor
from tabulate import tabulate

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "week2_db",
    "user": "student",
    "password": "student123"
}

def run_query(query, params=None):
    """Execute a SQL query and print results as a formatted table."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params)
            if cur.description:  # SELECT query
                rows = cur.fetchall()
                if rows:
                    headers = rows[0].keys()
                    print(tabulate([list(r.values()) for r in rows], headers=headers, tablefmt="grid"))
                    print(f"\n({len(rows)} row(s))")
                else:
                    print("No results.")
            else:
                conn.commit()
                print(f"Query executed. Rows affected: {cur.rowcount}")
    finally:
        conn.close()

### Quick Schema Reminder

Let's remind ourselves of the tables we are working with:

In [ ]:
run_query("""
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name;
""")

In [ ]:
run_query("""
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'employees'
ORDER BY ordinal_position;
""")

In [ ]:
run_query("""
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'departments'
ORDER BY ordinal_position;
""")

In [ ]:
run_query("""
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'sales'
ORDER BY ordinal_position;
""")

---

## 1. Basic GROUP BY

`GROUP BY` collapses multiple rows into summary rows. Every column in the `SELECT` that is **not** inside an aggregate function (`AVG`, `SUM`, `COUNT`, `MAX`, `MIN`) must appear in the `GROUP BY` clause.

### Example 1a: Average salary per department

In [ ]:
run_query("""
SELECT 
    department_id,
    ROUND(AVG(salary), 2) AS avg_salary
FROM employees
GROUP BY department_id
ORDER BY department_id;
""")

**What happened:**
- PostgreSQL grouped all employees by their `department_id`.
- For each group it computed `AVG(salary)`.
- We used `ROUND(..., 2)` to keep two decimal places.

### Example 1b: Count of employees per department

In [ ]:
run_query("""
SELECT 
    department_id,
    COUNT(*) AS employee_count
FROM employees
GROUP BY department_id
ORDER BY department_id;
""")

**Key point:** `COUNT(*)` counts every row in the group, including rows where columns might be NULL.

---

## 2. GROUP BY with JOIN

The `department_id` numbers are not very informative. Let us join with the `departments` table to show the actual department **name**.

In [ ]:
run_query("""
SELECT 
    d.department_name,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    COUNT(e.employee_id) AS employee_count
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_name
ORDER BY avg_salary DESC;
""")

**Notice:**
- We `GROUP BY d.department_name` because that is the non-aggregated column in our SELECT.
- We use `COUNT(e.employee_id)` instead of `COUNT(*)`. When there are no NULL employee_ids the result is the same, but `COUNT(column)` ignores NULLs while `COUNT(*)` does not (we will explore this difference in Section 5).

---

## 3. Multi-column GROUP BY

You can group by **multiple columns** to get finer-grained summaries. Let us compute total sales per region per month.

In [ ]:
run_query("""
SELECT 
    region,
    DATE_TRUNC('month', sale_date) AS sale_month,
    SUM(amount) AS total_sales,
    COUNT(*) AS num_transactions
FROM sales
WHERE region IS NOT NULL
GROUP BY region, DATE_TRUNC('month', sale_date)
ORDER BY region, sale_month;
""")

**Key points:**
- `DATE_TRUNC('month', sale_date)` converts any date to the first day of that month, effectively grouping all sales in the same calendar month together.
- We group by **both** `region` and the truncated month, producing one row per unique (region, month) pair.
- The `WHERE region IS NOT NULL` filter excludes NULL regions so we get clean output. We will revisit NULL handling in Section 5.

---

## 4. HAVING - Filtering Aggregated Results

`WHERE` filters rows **before** aggregation. `HAVING` filters groups **after** aggregation.

### Find departments with average salary above 75,000

In [ ]:
run_query("""
SELECT 
    d.department_name,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    COUNT(e.employee_id) AS employee_count
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_name
HAVING AVG(e.salary) > 75000
ORDER BY avg_salary DESC;
""")

**Why HAVING and not WHERE?**

| Clause | When it runs | Works on | Example |
|--------|-------------|----------|--------|
| `WHERE` | Before GROUP BY | Individual rows | `WHERE salary > 50000` |
| `HAVING` | After GROUP BY | Aggregated groups | `HAVING AVG(salary) > 75000` |

You **cannot** write `WHERE AVG(salary) > 75000` because the average does not exist yet at the point WHERE is evaluated.

---

## 5. Three Types of COUNT - Understanding NULLs

The `sales` table has a `region` column that may contain NULLs. Let us see how the three COUNT variants behave differently.

### `COUNT(*)` vs `COUNT(column)` vs `COUNT(DISTINCT column)`

In [ ]:
run_query("""
SELECT 
    COUNT(*) AS count_star,
    COUNT(region) AS count_region,
    COUNT(DISTINCT region) AS count_distinct_region
FROM sales;
""")

### What do these numbers mean?

| Count | Meaning | Includes NULLs? |
|-------|---------|----------------|
| `COUNT(*)` | Total number of rows in the table | Yes |
| `COUNT(region)` | Number of non-NULL values in region | No |
| `COUNT(DISTINCT region)` | Number of unique non-NULL regions | No |

The difference between `COUNT(*)` and `COUNT(region)` tells us **how many rows have a NULL region**. The difference between `COUNT(region)` and `COUNT(DISTINCT region)` tells us how many **duplicate** region values exist.

Let us verify the NULL count:

In [ ]:
run_query("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_regions,
    SUM(CASE WHEN region IS NOT NULL THEN 1 ELSE 0 END) AS non_null_regions
FROM sales;
""")

---

## 6. Simple CTE (Common Table Expression)

A CTE lets you define a named temporary result set using `WITH`. It makes queries much more readable than deeply nested subqueries.

### Problem: Find departments where the average salary is above the company-wide average

**Step 1:** Compute the company-wide average salary.

In [ ]:
run_query("""
SELECT ROUND(AVG(salary), 2) AS company_avg_salary
FROM employees;
""")

**Step 2:** Now use a CTE to combine both steps into one clean query.

In [ ]:
run_query("""
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal
    FROM employees
)
SELECT 
    d.department_name,
    ROUND(AVG(e.salary), 2) AS dept_avg_salary,
    ROUND(ca.avg_sal, 2) AS company_avg_salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
CROSS JOIN company_avg ca
GROUP BY d.department_name, ca.avg_sal
HAVING AVG(e.salary) > ca.avg_sal
ORDER BY dept_avg_salary DESC;
""")

**Why is this better?** Without the CTE you would need a nested subquery inside HAVING:

```sql
HAVING AVG(e.salary) > (SELECT AVG(salary) FROM employees)
```

The CTE names the concept ("company average") making the query self-documenting.

---

## 7. Multiple CTEs Chained Together

You can define **multiple CTEs** separated by commas. Each CTE can reference earlier CTEs. This is like building a pipeline.

### Problem: Get employee names from the top 3 departments by average salary

**Pipeline:**
1. CTE `dept_stats`: Compute average salary per department
2. CTE `top_departments`: Filter to the top 3 by average salary
3. Main query: Join back to employees to get names

In [ ]:
run_query("""
WITH dept_stats AS (
    -- Step 1: Compute department-level statistics
    SELECT 
        d.department_id,
        d.department_name,
        ROUND(AVG(e.salary), 2) AS avg_salary,
        COUNT(e.employee_id) AS employee_count
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    GROUP BY d.department_id, d.department_name
),
top_departments AS (
    -- Step 2: Keep only the top 3 departments
    SELECT 
        department_id,
        department_name,
        avg_salary,
        employee_count
    FROM dept_stats
    ORDER BY avg_salary DESC
    LIMIT 3
)
-- Step 3: Get all employees in those top departments
SELECT 
    td.department_name,
    e.first_name || ' ' || e.last_name AS full_name,
    e.salary,
    td.avg_salary AS dept_avg_salary
FROM top_departments td
JOIN employees e ON td.department_id = e.department_id
ORDER BY td.department_name, e.salary DESC;
""")

**Key insight:** Each CTE is like a variable in Python. You compute it once, name it, and reuse the name. This is far more readable than nesting subqueries three levels deep.

---

## 8. Correlated Subquery in WHERE

A **correlated subquery** references columns from the outer query. It runs once **per row** of the outer query.

### Problem: Find employees who earn more than the average salary of their own department

In [ ]:
run_query("""
SELECT 
    e.first_name || ' ' || e.last_name AS full_name,
    e.department_id,
    e.salary
FROM employees e
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.department_id = e.department_id
)
ORDER BY e.department_id, e.salary DESC;
""")

**How it works:**

For each employee row, the subquery computes the average salary **of that employee's department** (`WHERE e2.department_id = e.department_id`). If the employee's own salary exceeds that department average, the row is included.

This is a **correlated** subquery because the inner query references `e.department_id` from the outer query. It cannot run independently.

**Performance note:** Correlated subqueries can be slow on large tables because the inner query runs once per outer row. In some cases, rewriting with a CTE or JOIN is faster.

---

## 9. Subquery in FROM (Derived Table)

You can use a subquery in the `FROM` clause as if it were a table. This is called a **derived table**.

### Problem: For each department, show the department name and how far its average salary deviates from the company average

In [ ]:
run_query("""
SELECT 
    dept_avgs.department_name,
    dept_avgs.avg_dept_salary,
    ROUND(company_avg.avg_salary, 2) AS company_avg,
    ROUND(dept_avgs.avg_dept_salary - company_avg.avg_salary, 2) AS deviation
FROM (
    SELECT 
        d.department_name,
        AVG(e.salary) AS avg_dept_salary
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    GROUP BY d.department_name
) AS dept_avgs
CROSS JOIN (
    SELECT AVG(salary) AS avg_salary
    FROM employees
) AS company_avg
ORDER BY deviation DESC;
""")

**Why use a subquery in FROM?** Sometimes you need to aggregate first, then do further computation on those aggregates. The subquery produces an intermediate result that the outer query then works with.

We could rewrite this with CTEs (see next section) for even better readability.

---

## 10. Realistic Business Question - Built Step by Step

### The Request

Management wants a single report showing, **for each department**:
- Department name
- Number of employees
- Average salary
- Total sales revenue (from employees in that department)
- Name of the highest-paid employee

We will build this with multiple CTEs, then show the equivalent nested subquery approach.

### Approach: Multiple CTEs (Recommended for Readability)

In [ ]:
run_query("""
WITH dept_salary AS (
    -- CTE 1: Department-level salary statistics
    SELECT 
        d.department_id,
        d.department_name,
        COUNT(e.employee_id) AS employee_count,
        ROUND(AVG(e.salary), 2) AS avg_salary,
        ROUND(SUM(e.salary), 2) AS total_salary
    FROM departments d
    LEFT JOIN employees e ON d.department_id = e.department_id
    GROUP BY d.department_id, d.department_name
),
dept_sales AS (
    -- CTE 2: Total sales revenue per department
    SELECT 
        e.department_id,
        ROUND(SUM(s.amount), 2) AS total_revenue
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    GROUP BY e.department_id
),
top_earners AS (
    -- CTE 3: Find the highest-paid employee per department
    SELECT DISTINCT ON (e.department_id)
        e.department_id,
        e.first_name || ' ' || e.last_name AS top_earner_name,
        e.salary AS top_earner_salary
    FROM employees e
    ORDER BY e.department_id, e.salary DESC, e.employee_id
)
-- Final query: Combine all CTEs
SELECT 
    ds.department_name,
    ds.employee_count,
    ds.avg_salary,
    COALESCE(dv.total_revenue, 0) AS total_revenue,
    te.top_earner_name,
    te.top_earner_salary
FROM dept_salary ds
LEFT JOIN dept_sales dv ON ds.department_id = dv.department_id
LEFT JOIN top_earners te ON ds.department_id = te.department_id
ORDER BY ds.avg_salary DESC;
""")

**Why CTEs shine here:**

Each CTE solves one sub-problem:
- `dept_salary` handles employee counts and averages
- `dept_sales` handles revenue aggregation
- `top_earners` finds the highest-paid person using `DISTINCT ON`
- The final query simply joins the three results

Now let us see what the same query looks like with nested subqueries...

### Alternative: Nested Subquery Approach (Harder to Read)

In [ ]:
run_query("""
SELECT 
    d.department_name,
    COALESCE(emp_stats.employee_count, 0) AS employee_count,
    emp_stats.avg_salary,
    COALESCE(sales_rev.total_revenue, 0) AS total_revenue,
    top_emp.top_earner_name,
    top_emp.top_earner_salary
FROM departments d
LEFT JOIN (
    SELECT 
        department_id,
        COUNT(*) AS employee_count,
        ROUND(AVG(salary), 2) AS avg_salary
    FROM employees
    GROUP BY department_id
) emp_stats ON d.department_id = emp_stats.department_id
LEFT JOIN (
    SELECT 
        e.department_id,
        ROUND(SUM(s.amount), 2) AS total_revenue
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    GROUP BY e.department_id
) sales_rev ON d.department_id = sales_rev.department_id
LEFT JOIN (
    SELECT DISTINCT ON (department_id)
        department_id,
        first_name || ' ' || last_name AS top_earner_name,
        salary AS top_earner_salary
    FROM employees
    ORDER BY department_id, salary DESC, employee_id
) top_emp ON d.department_id = top_emp.department_id
ORDER BY emp_stats.avg_salary DESC NULLS LAST;
""")

### CTE vs Nested Subqueries: Comparison

| Aspect | CTE Approach | Nested Subquery Approach |
|--------|-------------|------------------------|
| **Readability** | High - each piece is named | Low - logic is buried in indentation |
| **Debugging** | Easy - test each CTE separately | Hard - must extract subqueries manually |
| **Reusability** | A CTE can be referenced multiple times | Each subquery is inlined |
| **Performance** | Similar (PostgreSQL often inlines CTEs) | Similar |
| **Preference** | Recommended for complex queries | Fine for simple, short subqueries |

**Rule of thumb:** If your subquery is more than 3-4 lines, consider a CTE.

---

## 11. Try It Yourself

Now it is your turn. Attempt each exercise before revealing the solution.

---

### Exercise 1: Monthly Revenue by Region with HAVING

Write a query that shows total sales per region per month, but **only** include region-month combinations where total revenue exceeds 50,000. Order by total revenue descending.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
SELECT 
    region,
    DATE_TRUNC('month', sale_date) AS sale_month,
    ROUND(SUM(amount), 2) AS total_revenue
FROM sales
WHERE region IS NOT NULL
GROUP BY region, DATE_TRUNC('month', sale_date)
HAVING SUM(amount) > 50000
ORDER BY total_revenue DESC;
```

</details>

In [ ]:
# Write your solution for Exercise 1 here
run_query("""
-- Your query here
""")

---

### Exercise 2: Employees Above Company Average Using CTE

Write a CTE-based query that lists all employees whose salary is above the company-wide average. Show their name, department name, salary, and how much above the company average they earn.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal
    FROM employees
)
SELECT 
    e.first_name || ' ' || e.last_name AS full_name,
    d.department_name,
    e.salary,
    ROUND(e.salary - ca.avg_sal, 2) AS above_average_by
FROM employees e
JOIN departments d ON e.department_id = d.department_id
CROSS JOIN company_avg ca
WHERE e.salary > ca.avg_sal
ORDER BY e.salary DESC;
```

</details>

In [ ]:
# Write your solution for Exercise 2 here
run_query("""
-- Your query here
""")

---

### Exercise 3: Department with Most Revenue

Write a query that finds the department with the **highest total sales revenue**. Show the department name, total revenue, and the number of sales transactions. Use CTEs.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
WITH dept_revenue AS (
    SELECT 
        d.department_name,
        ROUND(SUM(s.amount), 2) AS total_revenue,
        COUNT(s.sale_id) AS num_transactions
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    JOIN departments d ON e.department_id = d.department_id
    GROUP BY d.department_name
)
SELECT *
FROM dept_revenue
ORDER BY total_revenue DESC
LIMIT 1;
```

</details>

In [ ]:
# Write your solution for Exercise 3 here
run_query("""
-- Your query here
""")

---

## Summary Cheat Sheet

| Concept | Syntax | Use when |
|---------|--------|----------|
| **GROUP BY** | `SELECT col, AGG(x) FROM t GROUP BY col` | You need summary statistics per category |
| **HAVING** | `GROUP BY ... HAVING AGG(x) > value` | You need to filter after aggregation |
| **COUNT(*)** | `COUNT(*)` | Count all rows including NULLs |
| **COUNT(col)** | `COUNT(column_name)` | Count only non-NULL values |
| **COUNT DISTINCT** | `COUNT(DISTINCT col)` | Count unique non-NULL values |
| **CTE** | `WITH name AS (query) SELECT ... FROM name` | You want readable, modular queries |
| **Correlated subquery** | `WHERE col > (SELECT ... FROM t2 WHERE t2.x = t1.x)` | Per-row comparison with a related subset |
| **Derived table** | `FROM (SELECT ...) AS alias` | You need to aggregate then further process |

**Next steps:** Practice these patterns on your own datasets. The more you write GROUP BY and CTE queries, the more natural they become.